
## Hierarchical Clustering ##

<!-- ## Reduced measures using ward + euclidian distance - Threshold-based## 

IE branch colors were given here just to understand technically which branch do they normally belong to  -->

## Reduced measures for Threshold based using ward linkage + euclidian distance ##

IE branch colors were given to check normally which branch do they belong to. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from scipy.cluster.hierarchy import linkage, dendrogram, optimal_leaf_ordering, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from pyexpat import features

In [ ]:
metrics_path = "oath/to/igraph_metrics_summary.csv"
df = pd.read_csv (metrics_path, sep=",")

print(df.head())
print(df.shape)
print(df.columns.tolist())
print(df["method"].value_counts())

In [ ]:
#Language and branch mapping 
lang_map = {
    "bg": "Bulgarian",
    "is": "Icelandic",
    "ro": "Romanian",
    "ca": "Catalan",
    "lt": "Lithuanian",
    "de": "German",
    "sk": "Slovak",
    "no": "Norwegian",
    "pt": "Portuguese",
    "pl": "Polish",
    "da": "Danish",
    "mk": "Macedonian",
    "en": "English",
    "gl": "Galician",
    "lv": "Latvian",
    "fa": "Farsi",
    "uk": "Ukrainian",
    "sv": "Swedish",
    "sl": "Slovenian",
    "el": "Greek",
    "bs": "Bosnian",
    "bn": "Bengali",
    "cs": "Czech",
    "es": "Spanish",
    "ru": "Russian",
    "hr": "Croatian",
    "sq": "Albanian",
    "fr": "French",
    "nl": "Dutch",
    "sr": "Serbian",
    "si": "Sinhala",
    "it": "Italian"
}

branch_map = {
    "is": "Germanic",
    "de": "Germanic",
    "no": "Germanic",
    "da": "Germanic",
    "en": "Germanic",
    "sv": "Germanic",
    "nl": "Germanic",
    "ro": "Italic",
    "ca": "Italic",
    "pt": "Italic",
    "gl": "Italic",
    "es": "Italic",  
    "fr": "Italic",
    "it": "Italic",
    "bg": "Balto-Slavic",
    "lt": "Balto-Slavic",
    "sk": "Balto-Slavic",
    "pl": "Balto-Slavic",
    "lv": "Balto-Slavic",
    "uk": "Balto-Slavic",
    "sl": "Balto-Slavic",
    "bs": "Balto-Slavic",
    "cs": "Balto-Slavic",
    "ru": "Balto-Slavic",
    "hr": "Balto-Slavic",
    "sr": "Balto-Slavic",
    "mk": "Balto-Slavic",
    "fa": "Indo-Iranian",
    "bn": "Indo-Iranian",
    "si": "Indo-Iranian",
    "sq": "Albanian",
    "el": "Greek"
}

branch_palette = {
    "Germanic": "#C45B5B",
    "Italic": "#2F7FD3",
    "Balto-Slavic": "#2FAE66",
    "Indo-Iranian": "#D07A43",
    "Albanian": "#9A78D0",
    "Greek": "#C4AA3D"
}

df["language_full"] = df["language"].map(lang_map).fillna(df["language"])
df["branch"] = df["language"].map(branch_map)

# Threshold Method with Ward Linkage + Euclidian Distance 

In [ ]:
method_name = "thr_p95"  

df_thr = df[df["method"] == method_name].copy()

print("Number of threshold rows:", len(df_thr))
print(df_thr[["language", "language_full", "branch", "method"]].head())

features_thr = [
    "density",
    "degree_std",
    "max_degree",
    "largest_component_ratio",
    "diameter",
    "avg_clustering_coefficient",
    "transitivity",
    "modularity",
    "num_communities",
    "degree_assortativity"
]

X = df_thr[features_thr].copy()
labels = df_thr["language_full"].values

# For coloring labels by branch
label_to_branch = dict(zip(df_thr["language_full"], df_thr["branch"]))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) 


# Compute distance matrix using Euclidean distance
distance_matrix = pdist(X_scaled, metric="euclidean")

Z_ward = linkage(distance_matrix, method="ward")
Z = Z_ward

Z_ordered = optimal_leaf_ordering(Z, distance_matrix)

# Plot dendrogram with branch colored language labels
fig, ax = plt.subplots(figsize=(20, 12))

dendro = dendrogram(
    Z_ordered,
    labels=labels,
    leaf_rotation=75,
    leaf_font_size=25,
    color_threshold=0,
    above_threshold_color="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Ward linkage distance", fontsize=30, fontweight="bold")

for tick_label in ax.get_xmajorticklabels():
    lang_name = tick_label.get_text()
    branch = label_to_branch.get(lang_name)

    tick_label.set_color(branch_palette.get(branch, "black"))
    tick_label.set_fontweight("bold")
    tick_label.set_fontsize(20)
    tick_label.set_ha("right")

ax.tick_params(axis="y", labelsize=20)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")


for line in ax.get_lines():
    line.set_linewidth(1.6)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = [
    Patch(facecolor=color, edgecolor="none", label=branch)
    for branch, color in branch_palette.items()
]

legend = ax.legend(
    handles=legend_handles,
    title="Indo-European branch",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.40),
    ncol=3,
    frameon=True,
    fontsize=20,
    title_fontsize=23,
)

legend.get_title().set_fontweight("bold")

for text in legend.get_texts():
    text.set_fontweight("bold")

plt.subplots_adjust(bottom=0.38, top=0.96, left=0.08, right=0.98)
plt.show()


## Threshold Method without Albanian and Greek (forced 4 clusters)

In [ ]:

method_name = "thr_p95"

df_thr_no_ag = df[
    (df["method"] == method_name) &
    (~df["branch"].isin(["Albanian", "Greek"]))
].copy()

print("Number of threshold rows without Albanian and Greek:", len(df_thr_no_ag))

features_thr = [
    "density",
    "degree_std",
    "max_degree",
    "largest_component_ratio",
    "diameter",
    "avg_clustering_coefficient",
    "transitivity",
    "modularity",
    "num_communities",
    "degree_assortativity"
]

X_thr_no_ag = df_thr_no_ag[features_thr]

labels_thr_no_ag = df_thr_no_ag["language_full"].values

label_to_branch = dict(zip(df["language_full"], df["branch"]))

scaler = StandardScaler()
X_thr_no_ag_scaled = scaler.fit_transform(X_thr_no_ag)

distance_matrix_thr_no_ag = pdist(X_thr_no_ag_scaled, metric="euclidean")

Z_thr_no_ag = linkage(distance_matrix_thr_no_ag, method="ward")

Z_thr_no_ag_ordered = optimal_leaf_ordering(
    Z_thr_no_ag,
    distance_matrix_thr_no_ag
)


cluster_labels_thr_no_ag = fcluster(
    Z_thr_no_ag_ordered,
    t=4,
    criterion="maxclust"
)

df_thr_no_ag["cluster_4"] = cluster_labels_thr_no_ag

print(
    df_thr_no_ag[
        ["language", "language_full", "branch", "cluster_4"]
    ].sort_values("cluster_4")
)


fig, ax = plt.subplots(figsize=(20, 12))

dendro = dendrogram(
    Z_thr_no_ag_ordered,
    labels=labels_thr_no_ag,
    leaf_rotation=75,
    leaf_font_size=15,
    color_threshold=0,
    above_threshold_color="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Ward linkage distance", fontsize=20, fontweight="bold")

for tick_label in ax.get_xmajorticklabels():
    lang_name = tick_label.get_text()
    branch = label_to_branch.get(lang_name)

    tick_label.set_color(branch_palette.get(branch, "black"))
    tick_label.set_fontweight("bold")
    tick_label.set_fontsize(20)
    tick_label.set_ha("right")

ax.tick_params(axis="y", labelsize=20)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

for line in ax.get_lines():
    line.set_linewidth(1.6)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = [
    Patch(facecolor=color, edgecolor="none", label=branch)
    for branch, color in branch_palette.items()
    if branch not in ["Albanian", "Greek"]
]

legend = ax.legend(
    handles=legend_handles,
    title="Indo-European branch",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.40),
    ncol=2,
    frameon=True,
    fontsize=17,
    title_fontsize=20,
)

legend.get_title().set_fontweight("bold")

for text in legend.get_texts():
    text.set_fontweight("bold")

plt.subplots_adjust(bottom=0.38, top=0.96, left=0.08, right=0.98)

plt.show()

# Mutual kNN Method

In [ ]:
method_name = "mknn100"  

df_mknn = df[df["method"] == method_name].copy()

print("Number of mknn rows:", len(df_mknn))
print(df_mknn[["language", "language_full", "branch", "method"]].head())


features_mknn = [
    "density",
    "degree_std",
    "diameter",
    "avg_shortest_path",
    "avg_clustering_coefficient",
    "transitivity",
    "modularity",
    "num_communities",
    "degree_assortativity"
]

Y = df_mknn[features_mknn]

labels_mknn = df_mknn["language_full"].values

scaler = StandardScaler()
Y_scaled = scaler.fit_transform(Y)

distance_matrix = pdist(Y_scaled, metric="euclidean")

Y_ward = linkage(distance_matrix, method="ward")
Y = Y_ward

Y_ordered = optimal_leaf_ordering(Y, distance_matrix)
fig, ax = plt.subplots(figsize=(20, 12))

dendro = dendrogram(
    Y_ordered,
    labels=labels_mknn,
    leaf_rotation=75,
    leaf_font_size=25,
    color_threshold=0,
    above_threshold_color="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Ward linkage distance", fontsize=30, fontweight="bold")


for tick_label in ax.get_xmajorticklabels():
    lang_name = tick_label.get_text()
    branch = label_to_branch.get(lang_name)

    tick_label.set_color(branch_palette.get(branch, "black"))
    tick_label.set_fontweight("bold")
    tick_label.set_fontsize(20)
    tick_label.set_ha("right")

ax.tick_params(axis="y", labelsize=20)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")



for line in ax.get_lines():
    line.set_linewidth(1.6)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = [
    Patch(facecolor=color, edgecolor="none", label=branch)
    for branch, color in branch_palette.items()
]

legend = ax.legend(
    handles=legend_handles,
    title="Indo-European branch",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.40),
    ncol=3,
    frameon=True,
    fontsize=20,
    title_fontsize=23,
)

legend.get_title().set_fontweight("bold")

for text in legend.get_texts():
    text.set_fontweight("bold")


plt.subplots_adjust(bottom=0.38, top=0.96, left=0.08, right=0.98)
plt.show()

## Mutual kNN without Albanian and Italian (forced 4 clusters)

In [ ]:
method_name = "mknn100"

df_mknn_no_ag = df[
    (df["method"] == method_name) &
    (~df["branch"].isin(["Albanian", "Greek"]))
].copy()

print("Number of mKNN rows without Albanian and Greek:", len(df_mknn_no_ag))

features_mknn = [
    "density",
    "degree_std",
    "diameter",
    "avg_shortest_path",
    "avg_clustering_coefficient",
    "transitivity",
    "modularity",
    "num_communities",
    "degree_assortativity"
]

Y_mknn_no_ag = df_mknn_no_ag[features_mknn]

labels_mknn_no_ag = df_mknn_no_ag["language_full"].values


scaler = StandardScaler()
Y_mknn_no_ag_scaled = scaler.fit_transform(Y_mknn_no_ag)

distance_matrix_mknn_no_ag = pdist(Y_mknn_no_ag_scaled, metric="euclidean")

Z_mknn_no_ag = linkage(distance_matrix_mknn_no_ag, method="ward")

Z_mknn_no_ag_ordered = optimal_leaf_ordering(
    Z_mknn_no_ag,
    distance_matrix_mknn_no_ag
)

cluster_labels_mknn_no_ag = fcluster(
    Z_mknn_no_ag_ordered,
    t=4,
    criterion="maxclust"
)

df_mknn_no_ag["cluster_4"] = cluster_labels_mknn_no_ag

print(
    df_mknn_no_ag[
        ["language", "language_full", "branch", "cluster_4"]
    ].sort_values("cluster_4")
)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 12))

dendro = dendrogram(
    Z_mknn_no_ag_ordered,
    labels=labels_mknn_no_ag,
    leaf_rotation=75,
    leaf_font_size=15,
    color_threshold=0,
    above_threshold_color="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Ward linkage distance", fontsize=20, fontweight="bold")

for tick_label in ax.get_xmajorticklabels():
    lang_name = tick_label.get_text()
    branch = label_to_branch.get(lang_name)

    tick_label.set_color(branch_palette.get(branch, "black"))
    tick_label.set_fontweight("bold")
    tick_label.set_fontsize(20)
    tick_label.set_ha("right")

ax.tick_params(axis="y", labelsize=20)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

for line in ax.get_lines():
    line.set_linewidth(1.6)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = [
    Patch(facecolor=color, edgecolor="none", label=branch)
    for branch, color in branch_palette.items()
    if branch not in ["Albanian", "Greek"]
]

legend = ax.legend(
    handles=legend_handles,
    title="Indo-European branch",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.40),
    ncol=2,
    frameon=True,
    fontsize=17,
    title_fontsize=20,
)

legend.get_title().set_fontweight("bold")

for text in legend.get_texts():
    text.set_fontweight("bold")

plt.subplots_adjust(bottom=0.38, top=0.96, left=0.08, right=0.98)

plt.show()